# KSC rocket launch metrics development notebook

Notebook-first development workflow for computing NASA-relevant launch metrics from the KSC/CCSFS SDS archive.

Primary goals:

- read launch windows from the launch catalog
- read waveform windows from SDS
- apply StationXML response correction with ObsPy/FLOVOpy
- compute PGV, PGA, peak acoustic pressure, RMS, p99, spike/clipping QC metrics
- save raw windows, corrected quicklook plots, long-form metric CSVs, and optional EnhancedStream products

This notebook is intended to replace the split development between `compute_launch_peaks_physical.ipynb` and `compute_launch_peaks_physical.py`. It keeps the workflow notebook-driven while preserving the useful production features from the script.


## 1. Configuration

Edit this cell first. The paths below are intentionally explicit because this notebook is meant to be run interactively during development.

The most important knobs are:

- `events_csv`
- `stationxml`
- `sds_root`
- `outdir`
- `date_from`, `date_to`, and `max_launches`
- station/network/channel filters
- response-removal and filtering settings


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import pathlib
import re
import fnmatch
import json
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from obspy import UTCDateTime, read_inventory
from obspy.core.stream import Stream
from obspy.core.trace import Trace
from obspy.core.util.attribdict import AttribDict

# FLOVOpy imports.
# Keep these here so missing dependencies fail early and visibly.
#from flovopy.sds.sds import SDSobj
from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.enhanced.stream import EnhancedStream
from flovopy.core.preprocess import preprocess_stream
from flovopy.processing.spectrograms import icewebSpectrogram
#from flovopy.core.miniseed_io import read_mseed, write_mseed

# -----------------------------
# User-editable paths
# -----------------------------
events_csv = "all_florida_launches_with_seed_ids.csv"
#stationxml_dir = pathlib.Path.cwd().parent / "03_stationxml"
stationxml_dir = Path("/Volumes/haldata/KSC/station_metadata")
stationxml = stationxml_dir / "KSC.xml"
sds_root = Path("/Volumes/haldata/remastered/SDS_KSC")

# Output directory for products from this development notebook.
outdir = Path('~/work').expanduser() / "KSC_ensemble"
if not outdir.exists():
    outdir.mkdir(parents=True, exist_ok=True)
index_out = os.path.join(outdir, "20_launch_metric_products_index.csv")

# -----------------------------
# Run limits / event filters
# -----------------------------
date_from = None      # e.g. "2016-09-01" or None
date_to = None        # e.g. "2023-01-01" or None
max_launches = None   # e.g. 5 for testing

# -----------------------------
# Waveform selection
# -----------------------------
networks = ["*"]      # e.g. ["FL", "AM"] or ["*"]
stations = ["*"]      # e.g. ["BCHH", "BCH?"] or ["*"]
channels = ["*"]      # e.g. ["*HZ", "*DF", "*D*"] or ["*"]

# -----------------------------
# Processing settings
# -----------------------------
pad_s = 60.0          # read this much before/after each launch window
hp_hz = 0.4           # high-pass used during preprocessing
filter_type = "highpass"
verbose = 1

# If True, save raw padded windows before response correction.
save_raw_mseed = True

# If True, save corrected/trimmed MiniSEED after preprocessing.
save_corrected_mseed = True

# If True, attempt EnhancedStream.ampengfft() and save per-trace/station CSVs.
save_enhancedstream_products = True

# If True, save basic raw/corrected plots.
save_quicklook_plots = True

# If True, save spectrogram products from icewebSpectrogram.
save_spectrograms = True

# Optional pickle of EnhancedStream objects. Usually False for development.
save_pickle = False

# -----------------------------
# QC / robust amplitude settings
# -----------------------------
# These are NOT physical clipping limits unless you deliberately set them that way.
# They are used to compute robust/clipped QC metrics so one-sample spikes do not dominate.
SEIS_CLIP = None       # e.g. 0.01 m/s, or None to disable
INFRA_CLIP = None      # e.g. 3000.0 Pa, or None to disable
RMS_WIN_SEC = 1.0

Path(outdir).mkdir(parents=True, exist_ok=True)
print(f"Output directory: {Path(outdir).resolve()}")


## 2. Helper functions

In [ ]:

# -----------------------------
# Logging
# -----------------------------
def log(msg: str, verbose: int = 0, level: int = 1) -> None:
    if verbose >= level:
        print(msg, flush=True)


# -----------------------------
# General utilities
# -----------------------------
def parse_utc(s: Optional[str]) -> Optional[pd.Timestamp]:
    if s is None or s == "":
        return None
    return pd.to_datetime(s, utc=True, errors="coerce")


def window_overlaps(a0: pd.Timestamp, a1: pd.Timestamp, b0: Optional[pd.Timestamp], b1: Optional[pd.Timestamp]) -> bool:
    return (b0 is None or a1 >= b0) and (b1 is None or a0 <= b1)


def match_any(value: str, patterns: Iterable[str]) -> bool:
    pats = list(patterns) if patterns is not None else ["*"]
    return any(fnmatch.fnmatch(str(value), p) for p in pats)


def is_pressure_channel(channel: str) -> bool:
    """Heuristic channel classifier. KSC infrasound usually uses D* channels."""
    return len(str(channel)) >= 2 and str(channel)[1].upper() == "D"


def is_seismic_channel(channel: str) -> bool:
    """Heuristic channel classifier. KSC seismic channels are usually H* channels."""
    return len(str(channel)) >= 2 and str(channel)[1].upper() == "H"


def safe_slug(s: str, maxlen: int = 80) -> str:
    s = str(s) if s is not None else ""
    s = s.strip().replace("/", "-")
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return (s[:maxlen] or "launch")


def filter_stream_patterns(st: Stream,
                           network_patterns: Iterable[str] = ("*",),
                           station_patterns: Iterable[str] = ("*",),
                           channel_patterns: Iterable[str] = ("*",)) -> Stream:
    """Filter an ObsPy Stream using fnmatch patterns for network/station/channel."""
    out = Stream()
    for tr in st:
        if not match_any(tr.stats.network, network_patterns):
            continue
        if not match_any(tr.stats.station, station_patterns):
            continue
        if not match_any(tr.stats.channel, channel_patterns):
            continue
        out += tr
    return out


def ensure_utc(x: Any) -> UTCDateTime:
    """Convert pandas/string/ObsPy-ish time to UTCDateTime."""
    if isinstance(x, UTCDateTime):
        return x
    return UTCDateTime(pd.to_datetime(x, utc=True).to_pydatetime())


def event_basepath(row: pd.Series, launch_count: int, outdir: str) -> Tuple[str, str, str, str]:
    """Return slug, start_tag, eventdir, basepath for one launch row."""
    slug = safe_slug(row.get("slug", None) or row.get("name", None) or row.get("mission", None) or f"launch_{launch_count:04d}")
    start_tag = pd.to_datetime(row["window_start"], utc=True).strftime("%Y%m%dT%H%M%SZ")
    eventdir = os.path.join(outdir, start_tag)
    os.makedirs(eventdir, exist_ok=True)
    return slug, start_tag, eventdir, os.path.join(eventdir, slug)


# -----------------------------
# Flatten EnhancedStream metrics
# -----------------------------
def _flatten(d: Any, prefix: str = "") -> Dict[str, Any]:
    """Flatten nested dict/AttribDict objects into scalar key-value pairs."""
    out: Dict[str, Any] = {}

    if isinstance(d, AttribDict):
        d = dict(d)

    if isinstance(d, dict):
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            out.update(_flatten(v, key))
    elif isinstance(d, (list, tuple)):
        if len(d) == 0:
            out[prefix] = ""
        elif all(np.isscalar(x) or isinstance(x, str) for x in d):
            out[prefix] = json.dumps(list(d), default=str)
        else:
            out[prefix] = json.dumps(d, default=str)
    else:
        if np.isscalar(d) or isinstance(d, (str, bytes)) or d is None:
            out[prefix] = d
        else:
            out[prefix] = str(d)
    return out


def save_enhancedstream_bundle(est: EnhancedStream,
                               basepath: str,
                               include_pickle: bool = False,
                               make_plots: bool = True,
                               make_spectrograms: bool = True) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """
    Save EnhancedStream outputs:
      - corrected/processed MiniSEED
      - per-trace flattened CSV
      - station-level CSV if available
      - optional pickle
      - optional trace/spectrogram plots
    """
    mseed_path = basepath + "_corrected.mseed"
    trace_csv = basepath + "_trace_metrics.csv"
    station_csv = basepath + "_station_metrics.csv"
    pickle_path = basepath + ".pkl" if include_pickle else None

    try:
        est.write(mseed_path, format="MSEED")
    except Exception as e:
        log(f"  WARNING: could not write corrected MiniSEED {mseed_path}: {type(e).__name__}: {e}", verbose, 1)
        mseed_path = None

    # Per-trace metrics from tr.stats.metrics if present.
    rows = []
    for tr in est:
        row = {
            "id": tr.id,
            "network": tr.stats.network,
            "station": tr.stats.station,
            "location": tr.stats.location,
            "channel": tr.stats.channel,
            "starttime": str(tr.stats.starttime),
            "endtime": str(tr.stats.endtime),
            "sampling_rate": float(tr.stats.sampling_rate),
            "npts": int(tr.stats.npts),
        }
        metrics = getattr(tr.stats, "metrics", None)
        if metrics is not None:
            row.update(_flatten(metrics, "metrics"))
        rows.append(row)

    if rows:
        pd.DataFrame(rows).to_csv(trace_csv, index=False)
    else:
        trace_csv = None

    # Station-level metrics are FLOVOpy-version dependent. Try common locations.
    station_rows = []
    station_metrics = getattr(est, "station_metrics", None)
    if station_metrics is None:
        station_metrics = getattr(getattr(est, "stats", None), "station_metrics", None)

    if station_metrics is not None:
        if isinstance(station_metrics, pd.DataFrame):
            station_metrics.to_csv(station_csv, index=False)
        elif isinstance(station_metrics, dict):
            for station_key, metrics in station_metrics.items():
                row = {"station_key": station_key}
                row.update(_flatten(metrics, "metrics"))
                station_rows.append(row)
            pd.DataFrame(station_rows).to_csv(station_csv, index=False)
        else:
            pd.DataFrame([{"station_metrics_repr": repr(station_metrics)}]).to_csv(station_csv, index=False)
    else:
        station_csv = None

    if include_pickle and pickle_path is not None:
        try:
            import pickle
            with open(pickle_path, "wb") as f:
                pickle.dump(est, f)
        except Exception as e:
            log(f"  WARNING: could not pickle EnhancedStream: {type(e).__name__}: {e}", verbose, 1)
            pickle_path = None

    if make_plots:
        # Standard useful plot products from the script.
        plot_specs = [
            ("_Z_traces.png", dict(channel="*Z")),
            ("_P_traces.png", dict(channel="*D*")),
        ]
        for suffix, sel in plot_specs:
            try:
                sub = est.select(**sel)
                if len(sub):
                    sub.plot(outfile=basepath + suffix, size=(1200, 800), equal_scale=False)
            except Exception as e:
                log(f"  WARNING: plot failed {suffix}: {type(e).__name__}: {e}", verbose, 2)

        # Per-station trace plots.
        try:
            stations_here = sorted(set(tr.stats.station for tr in est))
            for sta in stations_here:
                sub = est.select(station=sta)
                if len(sub):
                    sub.plot(outfile=basepath + f"_{sta}_traces.png", size=(1200, 800), equal_scale=False)
        except Exception as e:
            log(f"  WARNING: per-station trace plots failed: {type(e).__name__}: {e}", verbose, 2)

    if make_spectrograms:
        # Conservative spectrogram products.
        spectro_specs = [
            ("_Z_spectrograms.png", dict(channel="*Z")),
            ("_P_spectrograms.png", dict(channel="*D*")),
        ]
        for suffix, sel in spectro_specs:
            try:
                sub = est.select(**sel)
                if len(sub):
                    icewebSpectrogram(sub).plot(outfile=basepath + suffix)
            except Exception as e:
                log(f"  WARNING: spectrogram failed {suffix}: {type(e).__name__}: {e}", verbose, 2)

    return mseed_path, trace_csv, station_csv


## 3. Robust metric functions

In [ ]:

def _safe_float_data(tr: Trace) -> np.ndarray:
    """Return float64 trace data for robust metric calculations."""
    return np.asarray(tr.data, dtype=np.float64)


def _clip_series(x: np.ndarray, clip_val: Optional[float]) -> Tuple[np.ndarray, float]:
    """
    Clip x to +/- clip_val. Return clipped array and fraction clipped.

    Use clip_val=None to disable clipping.
    """
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)

    if not finite.any():
        return x, np.nan

    if clip_val is None:
        return x, 0.0

    clipped = np.clip(x, -float(clip_val), float(clip_val))
    frac_clipped = float(np.mean(np.abs(x[finite]) > float(clip_val)))
    return clipped, frac_clipped


def _max_rolling_rms(x: np.ndarray, fs: float, win_sec: float = 1.0) -> float:
    """Maximum RMS in a sliding window."""
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if not finite.any():
        return np.nan

    x = np.nan_to_num(x, nan=0.0)
    nwin = max(1, int(round(float(win_sec) * float(fs))))

    if nwin >= x.size:
        return float(np.sqrt(np.mean(x ** 2)))

    x2 = x * x
    kernel = np.ones(nwin, dtype=np.float64) / nwin
    rms_series = np.sqrt(np.convolve(x2, kernel, mode="valid"))
    return float(np.max(rms_series))


def _dominant_frequency_fft(x: np.ndarray, fs: float,
                            fmin: Optional[float] = None,
                            fmax: Optional[float] = None) -> float:
    """Simple dominant frequency from single-sided amplitude spectrum."""
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if x.size < 4 or not finite.any():
        return np.nan

    x = np.nan_to_num(x - np.nanmean(x), nan=0.0)
    spec = np.abs(np.fft.rfft(x))
    freqs = np.fft.rfftfreq(x.size, d=1.0 / float(fs))

    # Exclude DC.
    mask = freqs > 0
    if fmin is not None:
        mask &= freqs >= float(fmin)
    if fmax is not None:
        mask &= freqs <= float(fmax)

    if not mask.any():
        return np.nan
    idxs = np.where(mask)[0]
    imax = idxs[np.argmax(spec[mask])]
    return float(freqs[imax])


def _mean_frequency_fft(x: np.ndarray, fs: float,
                        fmin: Optional[float] = None,
                        fmax: Optional[float] = None) -> float:
    """Power-weighted mean frequency from single-sided spectrum."""
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if x.size < 4 or not finite.any():
        return np.nan

    x = np.nan_to_num(x - np.nanmean(x), nan=0.0)
    amp = np.abs(np.fft.rfft(x))
    power = amp * amp
    freqs = np.fft.rfftfreq(x.size, d=1.0 / float(fs))

    mask = freqs > 0
    if fmin is not None:
        mask &= freqs >= float(fmin)
    if fmax is not None:
        mask &= freqs <= float(fmax)

    if not mask.any() or np.sum(power[mask]) <= 0:
        return np.nan
    return float(np.sum(freqs[mask] * power[mask]) / np.sum(power[mask]))


def _differentiate_for_pga(x: np.ndarray, fs: float) -> np.ndarray:
    """
    Estimate acceleration from velocity by numerical differentiation.

    This assumes the response-corrected seismic trace is velocity in m/s.
    Confirm StationXML/output units before using PGA values in the paper.
    """
    x = np.asarray(x, dtype=np.float64)
    if x.size < 2:
        return np.full_like(x, np.nan)
    return np.gradient(x, 1.0 / float(fs))


def trace_metrics(tr: Trace,
                  event_id: str,
                  slug: str,
                  t0: UTCDateTime,
                  t1: UTCDateTime,
                  clip_val: Optional[float] = None,
                  rms_win_sec: float = 1.0) -> Dict[str, Any]:
    """
    Return one long-form metric row for one event-channel trace.

    For seismic H* channels:
      - pgv_mps assumes corrected trace is velocity in m/s
      - pga_mps2 is numerical derivative of corrected velocity

    For infrasound D* channels:
      - peak_pressure_pa assumes corrected trace is pressure in Pa
    """
    fs = float(tr.stats.sampling_rate)
    x_raw = _safe_float_data(tr)
    x, frac_clipped = _clip_series(x_raw, clip_val)

    absx = np.abs(x)
    finite = np.isfinite(absx)

    peak_abs = float(np.nanmax(absx)) if finite.any() else np.nan
    p99_abs = float(np.nanpercentile(absx, 99)) if finite.any() else np.nan
    rms = float(np.sqrt(np.nanmean(x * x))) if finite.any() else np.nan
    max_1s_rms = _max_rolling_rms(x, fs, rms_win_sec)
    spike_ratio = float(peak_abs / p99_abs) if np.isfinite(p99_abs) and p99_abs > 0 else np.nan

    channel = str(tr.stats.channel)
    is_seis = is_seismic_channel(channel)
    is_infra = is_pressure_channel(channel)

    pgv_mps = peak_abs if is_seis else np.nan
    peak_pressure_pa = peak_abs if is_infra else np.nan

    if is_seis:
        acc = _differentiate_for_pga(x, fs)
        pga_mps2 = float(np.nanmax(np.abs(acc))) if np.isfinite(acc).any() else np.nan
    else:
        pga_mps2 = np.nan

    row = {
        "event_id": event_id,
        "slug": slug,
        "window_start": str(t0),
        "window_end": str(t1),
        "trace_id": tr.id,
        "network": tr.stats.network,
        "station": tr.stats.station,
        "location": tr.stats.location,
        "channel": tr.stats.channel,
        "sampling_rate_hz": fs,
        "npts": int(tr.stats.npts),
        "units_assumed": "m/s" if is_seis else ("Pa" if is_infra else "unknown"),
        "is_seismic": bool(is_seis),
        "is_infrasound": bool(is_infra),
        "peak_abs": peak_abs,
        "p99_abs": p99_abs,
        "rms": rms,
        "max_1s_rms": max_1s_rms,
        "spike_ratio_peak_over_p99": spike_ratio,
        "fraction_samples_clipped": frac_clipped,
        "pgv_mps": pgv_mps,
        "pga_mps2": pga_mps2,
        "peak_pressure_pa": peak_pressure_pa,
        "dominant_frequency_hz": _dominant_frequency_fft(x, fs, fmin=hp_hz),
        "mean_frequency_hz": _mean_frequency_fft(x, fs, fmin=hp_hz),
    }

    # Carry a few useful launch metadata fields through to the metric table later.
    return row


def station_quality_flag(metric_row: Dict[str, Any]) -> str:
    """
    Simple first-pass QC label.

    This is deliberately conservative and should be reviewed before publication.
    """
    if metric_row["npts"] < 2:
        return "no_data"
    if not np.isfinite(metric_row["peak_abs"]):
        return "bad"
    if np.isfinite(metric_row["fraction_samples_clipped"]) and metric_row["fraction_samples_clipped"] > 0.01:
        return "clipped"
    if np.isfinite(metric_row["spike_ratio_peak_over_p99"]) and metric_row["spike_ratio_peak_over_p99"] > 20:
        return "spiky"
    return "ok"


## 4. Load launch catalog and StationXML

In [ ]:
# Resolve and load launch catalog.
events_csv_path = Path(events_csv)
if not events_csv_path.exists():
    raise FileNotFoundError(f"events_csv not found: {events_csv_path.resolve()}")

df = pd.read_csv(events_csv_path)

# Required columns for this notebook.
required = {"window_start", "window_end"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"events_csv is missing required columns: {missing}")

df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
df["window_end"] = pd.to_datetime(df["window_end"], utc=True, errors="coerce")

date_from_ts = parse_utc(date_from)
date_to_ts = parse_utc(date_to)

if date_from_ts is not None or date_to_ts is not None:
    keep = []
    for _, r in df.iterrows():
        if pd.isna(r["window_start"]) or pd.isna(r["window_end"]):
            keep.append(False)
        else:
            keep.append(window_overlaps(r["window_start"], r["window_end"], date_from_ts, date_to_ts))
    df = df.loc[keep].copy()

if max_launches is not None:
    df = df.head(int(max_launches)).copy()

print(f"Loaded {len(df)} launch windows from {events_csv_path}")
display(df.head())
print(df.columns.tolist())


In [ ]:
# Load StationXML Inventory for response removal.
stationxml_path = Path(stationxml)
if not stationxml_path.exists():
    raise FileNotFoundError(f"StationXML not found: {stationxml_path}")

inv = read_inventory(str(stationxml_path))
print(inv)


In [ ]:
# Initialize SDS object.
if not sds_root.exists():
    raise FileNotFoundError(f"SDS root not found: {sds_root}")

sdsobject = EnhancedSDSClient(sds_root)
print(f"SDS root: {sds_root.resolve()}")


## 5. Stage A — read and save raw padded launch windows

This stage reads each launch window from SDS once, applies simple network/station/channel filters, and saves a raw padded MiniSEED file per event. This is useful because the raw read is often the slowest and most fragile step.

You can skip this stage if `index_out` already exists and points to valid raw MiniSEED files.


In [ ]:
def build_raw_window_index(force: bool = False) -> pd.DataFrame:
    """
    Read raw padded launch windows from SDS and write one raw MiniSEED file per launch.

    Returns an event index DataFrame.
    """
    if Path(index_out).exists() and not force:
        print(f"Reading existing index: {index_out}")
        event_df = pd.read_csv(index_out)
        return event_df

    index_rows = []
    launch_count = 0

    for _, row in df.iterrows():
        if pd.isna(row["window_start"]) or pd.isna(row["window_end"]):
            continue

        launch_count += 1
        t0 = ensure_utc(row["window_start"])
        t1 = ensure_utc(row["window_end"])
        tp0 = t0 - float(pad_s)
        tp1 = t1 + float(pad_s)

        slug, start_tag, eventdir, basepath = event_basepath(row, launch_count, outdir)
        event_id = start_tag

        log(f"\nLaunch {launch_count}: {slug}  {t0} -> {t1}", verbose, 1)

        try:
            sdsobject.read(tp0, tp1)
            st = sdsobject.stream
        except Exception as e:
            log(f"  READ FAIL: {type(e).__name__}: {e}", verbose, 1)
            index_rows.append({
                "event_id": event_id,
                "slug": slug,
                "window_start": str(t0),
                "window_end": str(t1),
                "status": "read_fail",
                "error": f"{type(e).__name__}: {e}",
                "basepath": basepath,
            })
            continue

        if not st or len(st) == 0:
            log("  NO DATA in window", verbose, 1)
            index_rows.append({
                "event_id": event_id,
                "slug": slug,
                "window_start": str(t0),
                "window_end": str(t1),
                "status": "no_data",
                "error": "no traces found",
                "basepath": basepath,
            })
            continue

        st = filter_stream_patterns(st, networks, stations, channels)

        if len(st) == 0:
            log("  NO DATA after network/station/channel filters", verbose, 1)
            index_rows.append({
                "event_id": event_id,
                "slug": slug,
                "window_start": str(t0),
                "window_end": str(t1),
                "status": "no_matching_traces",
                "error": "no traces after filters",
                "basepath": basepath,
            })
            continue

        raw_mseed_path = basepath + "_raw.mseed"

        if save_raw_mseed:
            try:
                write_mseed(st, raw_mseed_path)
            except Exception:
                # Fallback to standard ObsPy writer.
                st.write(raw_mseed_path, format="MSEED")

        # Carry through launch metadata that may be useful later.
        index_record = {
            "event_id": event_id,
            "slug": slug,
            "window_start": str(t0),
            "window_end": str(t1),
            "padded_start": str(tp0),
            "padded_end": str(tp1),
            "status": "raw_saved",
            "n_raw_traces": len(st),
            "basepath": basepath,
            "raw_mseed_path": raw_mseed_path if save_raw_mseed else "",
        }

        for col in ["name", "mission", "launch_designator", "SLC", "slc", "success",
                    "rocket", "vehicle", "payload", "operator", "landing", "booster_landing_type"]:
            if col in row.index:
                index_record[col] = row[col]

        index_rows.append(index_record)

    event_df = pd.DataFrame(index_rows)
    event_df.to_csv(index_out, index=False)
    print(f"\nWrote event index: {index_out}")
    return event_df


# Set force=True if you want to re-read SDS and overwrite the index.
event_df = build_raw_window_index(force=False)
display(event_df.head())
print(event_df["status"].value_counts(dropna=False))


In [ ]:
# --- 1-second RSAM source-steered launch detector prototype ---

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from obspy import Stream, UTCDateTime
from obspy.geodetics.base import gps2dist_azimuth

# Adjust if sam.py lives somewhere else in your repo
sys.path.insert(0, str(Path.cwd()))
from flovopy.processing.sam import RSAM


LAUNCHPADS_CSV = Path("launchpads.csv")
if not LAUNCHPADS_CSV.exists():
    LAUNCHPADS_CSV = Path("/mnt/data/launchpads.csv")

launchpads_df = pd.read_csv(LAUNCHPADS_CSV)
launchpads_df["pad"] = launchpads_df["pad"].astype(str).str.upper()


def get_trace_lat_lon(tr):
    """
    Try to recover station coordinates from trace metadata.
    Adapt this if your station coordinates live elsewhere.
    """
    coords = getattr(tr.stats, "coordinates", None)

    if coords is not None:
        lat = getattr(coords, "latitude", None)
        lon = getattr(coords, "longitude", None)
        if lat is not None and lon is not None:
            return float(lat), float(lon)

    for lat_key, lon_key in [
        ("latitude", "longitude"),
        ("lat", "lon"),
    ]:
        if hasattr(tr.stats, lat_key) and hasattr(tr.stats, lon_key):
            return float(getattr(tr.stats, lat_key)), float(getattr(tr.stats, lon_key))

    raise ValueError(f"No coordinates found for {tr.id}")


def compute_rsam_1s(st, freqmin=0.5, freqmax=20.0, metric="mean"):
    """
    Compute 1-second RSAM dataframes using existing RSAM class.
    RSAM is counts-based; no instrument correction required.
    """
    rsam = RSAM(
        stream=st.copy(),
        sampling_interval=1.0,
        filter=[freqmin, freqmax],
        bands=None,
        corners=4,
        despike=False,
        verbose=False,
    )

    frames = {}
    for seed_id, df in rsam.dataframes.items():
        if metric not in df.columns:
            continue
        d = df[["time", metric]].copy()
        d = d.rename(columns={metric: "value"})
        d["seed_id"] = seed_id
        frames[seed_id] = d

    return frames


def robust_normalize(x):
    """
    Normalize each RSAM trace before stacking so one huge channel does not dominate.
    """
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))

    if not np.isfinite(mad) or mad == 0:
        scale = np.nanstd(x)
    else:
        scale = 1.4826 * mad

    if not np.isfinite(scale) or scale == 0:
        return x * np.nan

    return (x - med) / scale


def stack_rsam_for_pad(
    rsam_frames,
    st_for_coords,
    pad_row,
    velocity_mps=340.0,
    metric_name="rsam_z",
):
    """
    Build one source-steered RSAM stack for one launchpad.

    The stack time is approximate source/origin time:
        origin_time = station_rsam_time - travel_time
    """
    pad = pad_row["pad"]
    pad_lat = float(pad_row["lat"])
    pad_lon = float(pad_row["lon"])

    tr_by_id = {tr.id: tr for tr in st_for_coords}

    shifted = []

    for seed_id, df in rsam_frames.items():
        if seed_id not in tr_by_id:
            continue

        try:
            sta_lat, sta_lon = get_trace_lat_lon(tr_by_id[seed_id])
        except ValueError:
            print(f"Skipping {seed_id}: no station coordinates")
            continue

        dist_m, _, _ = gps2dist_azimuth(pad_lat, pad_lon, sta_lat, sta_lon)
        tt_s = dist_m / velocity_mps

        d = df.copy()
        d["origin_time"] = d["time"] - tt_s
        d["value_norm"] = robust_normalize(d["value"].values)
        d["pad"] = pad
        d["distance_m"] = dist_m
        d["travel_time_s"] = tt_s
        shifted.append(d[["origin_time", "value_norm", "seed_id", "distance_m", "travel_time_s"]])

    if not shifted:
        return None, None

    all_shifted = pd.concat(shifted, ignore_index=True)

    # Round to 1-second bins in source-time coordinates
    all_shifted["origin_time_bin"] = np.round(all_shifted["origin_time"]).astype("int64")

    stack = (
        all_shifted
        .groupby("origin_time_bin")
        .agg(
            stack=("value_norm", "mean"),
            n=("value_norm", "count"),
            median_stack=("value_norm", "median"),
        )
        .reset_index()
        .rename(columns={"origin_time_bin": "time"})
    )

    stack["datetime"] = pd.to_datetime(stack["time"], unit="s", utc=True)
    stack["pad"] = pad

    return stack, all_shifted

def stack_rsam_for_pad(
    rsam_frames,
    st_for_coords,
    pad_row,
    inv_coords,
    velocity_mps=340.0,
):
    pad = pad_row["pad"]
    pad_lat = float(pad_row["lat"])
    pad_lon = float(pad_row["lon"])

    tr_by_id = {tr.id: tr for tr in st_for_coords}
    shifted = []

    for seed_id, df in rsam_frames.items():
        if seed_id not in tr_by_id:
            continue

        try:
            sta_lat, sta_lon = get_trace_lat_lon_from_inventory(
                tr_by_id[seed_id],
                inv_coords,
            )
        except ValueError:
            print(f"Skipping {seed_id}: no station coordinates")
            continue

        dist_m, _, _ = gps2dist_azimuth(pad_lat, pad_lon, sta_lat, sta_lon)
        tt_s = dist_m / velocity_mps

        d = df.copy()
        d["origin_time"] = d["time"] - tt_s
        d["value_norm"] = robust_normalize(d["value"].values)
        d["pad"] = pad
        d["distance_m"] = dist_m
        d["travel_time_s"] = tt_s
        shifted.append(d)

    if not shifted:
        return None, None

    all_shifted = pd.concat(shifted, ignore_index=True)
    all_shifted["origin_time_bin"] = np.round(all_shifted["origin_time"]).astype("int64")

    stack = (
        all_shifted
        .groupby("origin_time_bin")
        .agg(
            stack=("value_norm", "mean"),
            n=("value_norm", "count"),
            median_stack=("value_norm", "median"),
        )
        .reset_index()
        .rename(columns={"origin_time_bin": "time"})
    )

    stack["datetime"] = pd.to_datetime(stack["time"], unit="s", utc=True)
    stack["pad"] = pad

    return stack, all_shifted


def plot_pad_stacks(stacks, event_id=None, known_launch_time=None):
    plt.figure(figsize=(14, 5))

    for pad, stack in stacks.items():
        if stack is None or len(stack) == 0:
            continue
        plt.plot(stack["datetime"], stack["stack"], lw=1.0, label=pad)

    if known_launch_time is not None:
        t = pd.to_datetime(known_launch_time.datetime, utc=True)
        plt.axvline(t, color="k", ls="--", lw=1, label="known launch time")

    title = "Travel-time-corrected 1-s RSAM stacks"
    if event_id is not None:
        title += f": {event_id}"

    plt.title(title)
    plt.xlabel("Estimated source/origin time")
    plt.ylabel("Mean normalized RSAM stack")
    plt.legend(ncol=4)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def run_rsam_stack_for_event(
    row,
    sds,
    slc_col="slc",
    velocity_mps=340.0,
    freqmin=0.5,
    freqmax=20.0,
    metric="mean",
):
    """
    Read one full launch window, compute 1-s RSAM, and stack either:
      - only the known SLC if row[slc_col] exists and matches launchpads.csv
      - all SLCs otherwise
    """

    t0 = ensure_utc(row["window_start"])
    t1 = ensure_utc(row["window_end"])

    event_id = row.get("event_id", row.get("slug", row.name))
    print(f"\nRSAM stack for {event_id}: {t0} to {t1}")

    st = sds.read(
        starttime=t0,
        endtime=t1,
        net=networks,
        sta=stations,
        loc="00",
        chan=channels,
        skip_low_rate_channels=True,
        merge=None,
        trim=True,
        daywise=False,
        postprocess=False,
        final_smart_merge=False,
        verbose=False,
    )

    if len(st) == 0:
        print("  No waveform data")
        return None

    st = filter_stream_patterns(st, networks, stations, channels)

    # Keep only normal infrasound + seismic channels for now
    st = Stream([
        tr for tr in st
        if tr.stats.channel.upper() in {"DD1", "DD2", "DD3", "DHE", "DHN", "DHZ"}
    ])

    if len(st) == 0:
        print("  No usable channels after filtering")
        return None

    print(f"  raw traces for RSAM: {len(st)}")

    rsam_frames = compute_rsam_1s(
        st,
        freqmin=freqmin,
        freqmax=freqmax,
        metric=metric,
    )

    if not rsam_frames:
        print("  RSAM produced no usable traces")
        return None

    print(f"  RSAM traces: {len(rsam_frames)}")

    known_slc = None
    if slc_col in row and pd.notna(row[slc_col]):
        known_slc = str(row[slc_col]).upper().strip()

    if known_slc and known_slc in set(launchpads_df["pad"]):
        pads_to_try = launchpads_df[launchpads_df["pad"] == known_slc]
        print(f"  stacking known SLC: {known_slc}")
    else:
        pads_to_try = launchpads_df
        print("  no matching SLC in metadata; stacking all pads")

    stacks = {}
    shifted_by_pad = {}

    for _, pad_row in pads_to_try.iterrows():
        stack, shifted = stack_rsam_for_pad(
            rsam_frames,
            st,
            pad_row,
            inv_coords=inv_coords,
            velocity_mps=velocity_mps,
        )
        stacks[pad_row["pad"]] = stack
        shifted_by_pad[pad_row["pad"]] = shifted

    known_launch_time = None
    for col in ["launch_time", "t0", "event_time", "datetime"]:
        if col in row and pd.notna(row[col]):
            known_launch_time = ensure_utc(row[col])
            break

    plot_pad_stacks(
        stacks,
        event_id=event_id,
        known_launch_time=known_launch_time,
    )

    return {
        "event_id": event_id,
        "stream": st,
        "rsam_frames": rsam_frames,
        "stacks": stacks,
        "shifted_by_pad": shifted_by_pad,
    }


# Example: test on events 2 and 3 from your current event dataframe.
# Adjust event_df name if yours is different.
print(event_df.columns.tolist())

test_results = []

for idx in [1, 2]:
    result = run_rsam_stack_for_event(
        event_df.iloc[idx],
        sds=sdsobject,
        slc_col="slc",          # change to "pad" or "launchpad" if needed
        velocity_mps=340.0,
        freqmin=0.5,
        freqmax=20.0,
        metric="mean",
    )
    test_results.append(result)

## 6. Stage B — preprocess, response-correct, and compute metrics

This stage reads the raw MiniSEED windows from Stage A, applies response correction using StationXML, trims to the exact launch window, computes long-form per-channel metrics, and saves both CSV and optional quicklook products.

The long-form metrics table is the prototype of the future SQLite `event_channel_metrics` table.


In [ ]:
def preprocess_one_event(row: pd.Series, sds: EnhancedSDSClient) -> Optional[EnhancedStream]:
    """Read one event directly from SDS, trim tightly, preprocess, return corrected EnhancedStream."""
    t0 = ensure_utc(row["window_start"])
    t1 = ensure_utc(row["window_end"])

    if t1 <= t0:
        print(f"  Bad/zero window: {t0} to {t1}")
        return None

    st = sds.read(
        starttime=t0,
        endtime=t1,
        net=networks,
        sta=stations,
        loc="00",
        chan=channels,
        skip_low_rate_channels=True,
        merge=None,
        trim=True,
        daywise=False,
        postprocess=False,
        final_smart_merge=False,
        verbose=False,
    )

    if st is None or len(st) == 0:
        return None

    st = filter_stream_patterns(st, networks, stations, channels)
    st.trim(t0, t1, pad=False)

    if len(st) == 0:
        return None

    st_pp = preprocess_stream(
        st,
        freq=hp_hz,
        filter_type=filter_type,
        inv=inv,
        verbose=verbose,
    )

    if st_pp is None or len(st_pp) == 0:
        return None

    st_pp.trim(t0, t1, pad=False)

    if len(st_pp) == 0 or all(tr.stats.npts < 2 for tr in st_pp):
        return None

    return st_pp if isinstance(st_pp, EnhancedStream) else EnhancedStream(st_pp)


def metrics_rows_from_enhancedstream(est: EnhancedStream, row: pd.Series, t0, t1) -> list[dict]:
    """Convert EnhancedStream trace metrics to long-form table rows."""
    rows = []

    for tr in est:
        m = dict(getattr(tr.stats, "metrics", {}) or {})

        mrow = {
            "event_id": row["event_id"],
            "slug": row["slug"],
            "seed_id": tr.id,
            "network": tr.stats.network,
            "station": tr.stats.station,
            "location": tr.stats.location,
            "channel": tr.stats.channel,
            "component": tr.stats.channel[-1] if tr.stats.channel else None,
            "units": tr.stats.get("units", None),
            "starttime": str(tr.stats.starttime),
            "endtime": str(tr.stats.endtime),
            "sampling_rate": float(tr.stats.sampling_rate),
            "npts": int(tr.stats.npts),
            "window_start": str(t0),
            "window_end": str(t1),
            **m,
        }

        for col in [
            "name", "mission", "launch_designator", "SLC", "slc", "success",
            "rocket", "vehicle", "payload", "operator", "landing",
            "booster_landing_type",
        ]:
            if col in row.index:
                mrow[col] = row[col]

        rows.append(mrow)

    return rows


def compute_metrics_for_events(event_df: pd.DataFrame, sds: EnhancedSDSClient, force: bool = False) -> pd.DataFrame:
    """Compute long-form channel metrics for all valid event rows."""
    metrics_out = os.path.join(outdir, "event_channel_metrics_long.csv")

    if Path(metrics_out).exists() and not force:
        print(f"Reading existing metrics table: {metrics_out}")
        return pd.read_csv(metrics_out)

    all_rows = []
    product_rows = []

    valid_events = event_df[
        event_df["window_start"].notna() & event_df["window_end"].notna()
    ].copy()

    print(f"Computing metrics for {len(valid_events)} events")

    for n, (_, row) in enumerate(valid_events.iterrows(), start=1):
        event_id = row["event_id"]
        slug = row["slug"]
        basepath = row.get("basepath", os.path.join(outdir, f"{event_id}_{slug}"))

        t0 = ensure_utc(row["window_start"])
        t1 = ensure_utc(row["window_end"])

        print(f"\n{n}/{len(valid_events)} {event_id} {slug}  ({(t1 - t0):.1f} s)")

        try:
            est = preprocess_one_event(row, sds)
        except Exception as e:
            print(f"  preprocess FAILED: {type(e).__name__}: {e}")
            continue

        if est is None or len(est) == 0:
            print("  No usable corrected data")
            continue

        print(f"  corrected traces: {len(est)}")

        try:
            est.ampengfft(verbose=False)
        except Exception as e:
            print(f"  ampengfft WARNING: {type(e).__name__}: {e}")

        all_rows.extend(metrics_rows_from_enhancedstream(est, row, t0, t1))

        if save_enhancedstream_products:
            try:
                mseed_path, trace_csv, station_csv = save_enhancedstream_bundle(
                    est,
                    basepath=basepath,
                    include_pickle=save_pickle,
                    make_plots=save_quicklook_plots,
                    make_spectrograms=save_spectrograms,
                )
                product_rows.append({
                    "event_id": event_id,
                    "slug": slug,
                    "corrected_mseed_path": mseed_path,
                    "trace_metrics_csv": trace_csv,
                    "station_metrics_csv": station_csv,
                    "basepath": basepath,
                })
            except Exception as e:
                print(f"  EnhancedStream save WARNING: {type(e).__name__}: {e}")

        elif save_corrected_mseed:
            try:
                est.write(basepath + "_corrected.mseed", format="MSEED")
            except Exception as e:
                print(f"  corrected MiniSEED write WARNING: {type(e).__name__}: {e}")

    metrics_df = pd.DataFrame(all_rows)
    metrics_df.to_csv(metrics_out, index=False)
    print(f"\nWrote long-form metrics: {metrics_out}")

    if product_rows:
        products_df = pd.DataFrame(product_rows)
        products_out = os.path.join(outdir, "event_products_index.csv")
        products_df.to_csv(products_out, index=False)
        print(f"Wrote products index: {products_out}")

    return metrics_df


save_enhancedstream_products = False
save_quicklook_plots = False
save_spectrograms = False
save_corrected_mseed = False

event_df["window_start_dt"] = pd.to_datetime(event_df["window_start"], errors="coerce")
event_df["window_end_dt"] = pd.to_datetime(event_df["window_end"], errors="coerce")
event_df["window_duration_s"] = (
    event_df["window_end_dt"] - event_df["window_start_dt"]
).dt.total_seconds()

event_df2 = (
    event_df[event_df["window_duration_s"].between(60, 7200)]
    .head(5)
    .copy()
)

metrics_df = compute_metrics_for_events(event_df2, sdsobject, force=True)

display(metrics_df.head())
print(metrics_df.shape)
display(event_df2[["event_id", "slug", "window_start", "window_end", "window_duration_s"]])


## 7. Derived wide tables for quick inspection

The long-form table above is the future database-friendly format. These wide tables mimic the exploratory notebook outputs and are handy for fast visual inspection.


In [ ]:
def write_wide_metric_tables(metrics_df: pd.DataFrame, outdir: str) -> None:
    """Export wide event × trace tables for selected metrics."""
    Path(outdir).mkdir(parents=True, exist_ok=True)

    specs = [
        ("pgv_mps", "pgv_peak_abs.csv", "is_seismic"),
        ("pga_mps2", "pga_peak_abs.csv", "is_seismic"),
        ("peak_pressure_pa", "pap_peak_abs.csv", "is_infrasound"),
        ("p99_abs", "p99_abs.csv", None),
        ("max_1s_rms", "max_1s_rms.csv", None),
        ("spike_ratio_peak_over_p99", "spike_ratio_peak_over_p99.csv", None),
        ("fraction_samples_clipped", "fraction_samples_clipped.csv", None),
        ("dominant_frequency_hz", "dominant_frequency_hz.csv", None),
        ("mean_frequency_hz", "mean_frequency_hz.csv", None),
    ]

    for metric, filename, flag_col in specs:
        if metric not in metrics_df.columns:
            continue

        d = metrics_df.copy()
        if flag_col is not None and flag_col in d.columns:
            d = d[d[flag_col].astype(bool)]

        if d.empty:
            continue

        wide = d.pivot_table(
            index=["event_id", "slug", "window_start", "window_end"],
            columns="trace_id",
            values=metric,
            aggfunc="max",
        ).reset_index()

        # Flatten pandas column index if needed.
        wide.columns = [str(c) for c in wide.columns]
        out = Path(outdir) / filename
        wide.to_csv(out, index=False)
        print(f"Wrote {out}")


write_wide_metric_tables(metrics_df, outdir)


## 8. Quick summaries for the NASA-relevant paper

These are early sanity checks, not final paper plots.


In [ ]:
# Event-level maxima.
if not metrics_df.empty:
    event_summary = metrics_df.groupby(["event_id", "slug"], dropna=False).agg(
        max_pgv_mps=("pgv_mps", "max"),
        max_pga_mps2=("pga_mps2", "max"),
        max_peak_pressure_pa=("peak_pressure_pa", "max"),
        n_channels=("trace_id", "nunique"),
        n_stations=("station", "nunique"),
    ).reset_index()

    summary_out = Path(outdir) / "event_summary_maxima.csv"
    event_summary.to_csv(summary_out, index=False)
    print(f"Wrote {summary_out}")
    display(event_summary.sort_values("max_peak_pressure_pa", ascending=False).head(20))
else:
    print("metrics_df is empty")


In [ ]:
# Station-level maxima / medians.
if not metrics_df.empty:
    station_summary = metrics_df.groupby(["station"], dropna=False).agg(
        max_pgv_mps=("pgv_mps", "max"),
        median_pgv_mps=("pgv_mps", "median"),
        max_pga_mps2=("pga_mps2", "max"),
        median_pga_mps2=("pga_mps2", "median"),
        max_peak_pressure_pa=("peak_pressure_pa", "max"),
        median_peak_pressure_pa=("peak_pressure_pa", "median"),
        n_events=("event_id", "nunique"),
        n_channels=("trace_id", "nunique"),
    ).reset_index()

    station_out = Path(outdir) / "station_summary_metrics.csv"
    station_summary.to_csv(station_out, index=False)
    print(f"Wrote {station_out}")
    display(station_summary.sort_values("max_peak_pressure_pa", ascending=False).head(20))
else:
    print("metrics_df is empty")


## 9. Next refactor target

Once this notebook is stable, the natural split is:

```text
04_event_catalog/
  notebooks/
    compute_launch_metrics_dev.ipynb
  ksc_catalog/
    io.py          # SDS, StationXML, launch CSV loading
    metrics.py     # trace_metrics, robust QC metrics
    products.py    # plots, spectrograms, CSV/MSEED outputs
    db.py          # later SQLite layer
```

For now, this notebook is the main development driver.
